# 第8回　データの加工：並べ替え・ランキング
## ―― ランキングは何を見せて、何を隠すか

情報活用Ⅰ　／　北星学園大学　2026年度後期

ランキングは、いちばん作りたくなる表である。そして、いちばん誤解を生む表でもある。

In [ ]:
# 準備：ライブラリと、練習用データ（北辰大学の学生400人）を読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    # ネットから取れないときは、同じデータをその場で作る（中身は気にしなくてよい）
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("読み込めた行数:", len(df))
df.head()

---
## 1. 学部別テスト点ランキングを作る

`groupby` で学部ごとにまとめ、`sort_values` で並べ替える。

In [ ]:
rank_tbl = (df.groupby("学部")["テスト点"]
              .mean()
              .sort_values(ascending=False)
              .round(1)
              .reset_index())
rank_tbl.index = rank_tbl.index + 1        # 1位から始める
rank_tbl.columns = ["学部", "平均点"]
rank_tbl

順位表ができた。**「1位・2位・3位」と並ぶと、はっきり差があるように見える。**

この表を新聞に載せたら、見出しはこうなるだろう ―― 「経済学部が首位」。

では、実際の差はどれくらいか。

In [ ]:
top = rank_tbl["平均点"].iloc[0]
bottom = rank_tbl["平均点"].iloc[-1]
print(f"1位: {rank_tbl['学部'].iloc[0]}  {top} 点")
print(f"3位: {rank_tbl['学部'].iloc[-1]}  {bottom} 点")
print()
print(f"1位と3位の差: {top - bottom:.1f} 点")
print(f"テスト点全体の標準偏差（ばらつき）: {df['テスト点'].std():.1f} 点")
print()
print(f"→ 順位1位と3位の差は、個人差の {(top - bottom) / df['テスト点'].std():.2f} 倍でしかない")

**1位と3位の差は約2点。個人のばらつきは約10点。**

つまり、**同じ学部の中の人と人の違いのほうが、学部と学部の違いよりはるかに大きい。**
しかし順位表には、それが1行も書かれていない。

### 目で見る ―― 順位表と、実際の分布

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# 左：順位表の印象（平均値だけの棒グラフ・Y軸は0から）
ax[0].bar(rank_tbl["学部"], rank_tbl["平均点"], color="#80cbc4", edgecolor="white")
ax[0].set_ylim(0, 70)
ax[0].set_ylabel("平均点")
ax[0].set_title("平均値だけを見た場合")

# 右：一人ひとりの点を重ねる
for i, gk in enumerate(rank_tbl["学部"]):
    v = df[df["学部"] == gk]["テスト点"]
    ax[1].scatter(np.random.normal(i, 0.08, len(v)), v, alpha=0.25, s=14, color="#1565c0")
    ax[1].hlines(v.mean(), i - 0.3, i + 0.3, color="#e8503a", lw=3)
ax[1].set_xticks(range(len(rank_tbl)))
ax[1].set_xticklabels(rank_tbl["学部"])
ax[1].set_ylabel("テスト点")
ax[1].set_title("一人ひとりを重ねた場合（赤線が平均）")

plt.tight_layout(); plt.show()

右のグラフを見れば分かる。**3つの山は、ほとんど重なっている。**
赤い平均線の高さの違いは、点の散らばりに比べて小さい。

> **順位は差の大きさを消す。** 1位と2位が僅差でも、順位表では等間隔に見える。
> 順位表を作ったら、**必ず実数も併記する。**

---
## 2. 並べ替えと順位付けの書き方

使う道具は2つ。

| 関数 | 何をするか |
|---|---|
| `sort_values()` | 行を並べ替える |
| `rank()` | 順位の数字（1, 2, 3…）をつける |

In [ ]:
# テスト点が高い順に、上位10人
df.sort_values("テスト点", ascending=False).head(10)[["学生ID", "学部", "テスト点", "勉強時間h"]]

In [ ]:
# 複数のキーで並べ替える（学部ごとに、点の高い順）
df.sort_values(["学部", "テスト点"], ascending=[True, False]).head(8)[["学生ID", "学部", "テスト点"]]

In [ ]:
# 順位の数字をつける
tmp = df[["学生ID", "学部", "テスト点"]].copy()
tmp["全体順位"] = tmp["テスト点"].rank(ascending=False, method="min").astype(int)
tmp["学部内順位"] = tmp.groupby("学部")["テスト点"].rank(ascending=False, method="min").astype(int)
tmp.sort_values("全体順位").head(10)

`method="min"` は同点のときの扱い。同じ点なら同じ順位（1位が2人なら次は3位）になる。
**同点が多いデータで順位をつけると、順位はほとんど意味を持たなくなる。**

In [ ]:
# 同点はどれくらいあるか
dup = df["テスト点"].value_counts()
print(f"同じ点の人がいる点数: {(dup > 1).sum()} 通り")
print(f"最も多い同点: {dup.max()} 人が {dup.idxmax()} 点")

---
## 3. 少ない人数のランキングは何を意味するか

第6回で見たとおり、**人数が少ないほど平均はぶれる。**
回答が3人しかいない学年を1位として並べたら、その1位は何を意味するのか。

In [ ]:
# 3人だけのグループの平均が、どれくらいぶれるか
scores = df["テスト点"]
print("3人だけで平均を出す（8回くり返す）")
for i in range(8):
    print(f"  {i+1}回目: {scores.sample(3).mean():.1f} 点")
print()
print(f"（400人全員の平均は {scores.mean():.1f} 点）")

**同じ集団から取り出しているのに、1位にも最下位にもなる。**

あなたの調査でも、選択肢によっては回答が数人しかない項目が出る。そこに順位をつけて「◯◯が1位」と書くと、**偶然を発見として報告することになる。**

対処は難しくない。**表に人数（n）を必ず併記する。** 読む人が自分で判断できるようになる。

In [ ]:
# 人数を併記した表（この形で報告書に載せる）
safe = (df.groupby("学部")["テスト点"]
          .agg(人数="count", 平均="mean", 標準偏差="std")
          .round(1)
          .sort_values("平均", ascending=False))
safe

---
## 4. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# ランキングを作る（列名を書き換える）
# mydf.groupby("グループの列")["数値の列"].agg(人数="count", 平均="mean", 標準偏差="std").round(1).sort_values("平均", ascending=False)

---
## 5. 卒業課題 ―― ランキングを使わずに同じ主張ができるか

作ったランキングで言いたいことを、**順位を使わずに**言い直してみる。
実数で示す、グラフで示す、差の大きさを個人差と比べる。

言い直せたなら、そのランキングは主張の役に立っている。言い直せないなら、**順位という見せ方だけが主張を作っていた**ということになる。

---
## 課題8（6点）

**このノートブック** ＋ **ランキング表** ＋ **「この順位表では言えないこと」3つ**。

- [ ] 自分のデータのランキング表（**人数を併記すること**）
- [ ] この順位表では言えないこと、3つ
- [ ] （書ければ）順位を使わずに同じ主張をした場合の書き方

「言えないこと」の例：1位と2位の差が意味のある差か／人数が少ない項目が混ざっていないか／他の要因で分けたら順位が変わらないか（第7回）

提出期限：次回授業の開始まで（遅れた場合は50%）